In [3]:
import torch
import torch.nn.functional as F
from torch.utils.data import random_split
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch import nn, optim

In [53]:
class NN(nn.Module):
    def __init__(self, input_size, num_classes: int, hidden_unit: int):
        super().__init__()
        self.linear_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size, hidden_unit),
            nn.Linear(hidden_unit, num_classes)
        )
    def forward(self, x):
        x = self.linear_stack(x)
        return x
model0 = NN(input_size = 784, num_classes = 10, hidden_unit = 50)
model0

NN(
  (linear_stack): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=50, bias=True)
    (2): Linear(in_features=50, out_features=10, bias=True)
  )
)

In [9]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [13]:
batch_size = 32
learning_rate = 0.001
input_size = 784
num_classes = 10
epochs = 10
import os
num_workers = os.cpu_count()

In [21]:
from pathlib import Path
data_dir = Path('data')
entire_train_dataset = datasets.MNIST(
    data_dir,
    transform = transforms.ToTensor(),
    train = True
)
test_dataset = datasets.MNIST(
    data_dir,
    transform = transforms.ToTensor(),
    train = False
)
train_dataset, val_dataset = random_split(entire_train_dataset, [50000,10000])
len(train_dataset), len(val_dataset)

(50000, 10000)

In [56]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size = batch_size,
    pin_memory = True,
    num_workers = num_workers,
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size = batch_size,
    pin_memory = True,
    num_workers = num_workers,
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size = batch_size,
    pin_memory = True,
    num_workers = num_workers,
)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model0.parameters(), lr = learning_rate)
from tqdm.auto import tqdm
for epoch in tqdm(range(epochs)):
    for batch, (X,y) in enumerate(train_dataloader):
        model0.train()
        y_logits = model0(X)
        loss = loss_fn(y_logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        

  0%|          | 0/10 [00:00<?, ?it/s]

In [82]:
next(model0.parameters())

Parameter containing:
tensor([[ 0.0344, -0.0248,  0.0081,  ...,  0.0158, -0.0095,  0.0098],
        [ 0.0198,  0.0323,  0.0357,  ...,  0.0012, -0.0301, -0.0169],
        [-0.0147,  0.0234, -0.0276,  ...,  0.0070,  0.0083,  0.0296],
        ...,
        [-0.0044, -0.0277, -0.0262,  ...,  0.0168, -0.0235,  0.0066],
        [ 0.0040, -0.0016,  0.0278,  ..., -0.0083, -0.0284, -0.0152],
        [ 0.0244,  0.0123,  0.0201,  ..., -0.0089,  0.0239, -0.0245]],
       requires_grad=True)

In [90]:
def check_accuracy(model, dataloader):
    num_correct = 0
    num_samples = 0
    model.eval()
    with torch.no_grad():
        for X,y in dataloader:
            X = X.to(device)
            y = y.to(device)
            logits = model(X)
            loss = loss_fn(logits, y)
            _, predictions = logits.max(1)
            num_correct += (predictions==y).sum()
            num_samples += predictions.size(0)
    model.train()
    return num_correct/num_samples

In [92]:
print(f"Accuracy on training set: {check_accuracy(model0, train_dataloader)*100:.2f}")
print(f"Accuracy on validation set: {check_accuracy(model0, val_dataloader)*100:.2f}")
print(f"Accuracy on test set: {check_accuracy(model0, test_dataloader)*100:.2f}")

Accuracy on training set: 93.17
Accuracy on validation set: 91.44
Accuracy on test set: 92.21


In [94]:
import pytorch_lightning as pl

In [174]:
import torchmetrics
class NN(pl.LightningModule):
    def __init__(self, input_size=784, hidden_units=50, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_units)
        self.fc2 = nn.Linear(hidden_units, num_classes)
        self.loss = nn.CrossEntropyLoss()
        # self.accuracy = torchmetrics.Accuracy(task = 'Multi-class', num_classes=num_classes)
        # self.myaccuracy = MyAccuracy()
        # self.F1score = torchmetrics.F1Score(task = 'Multi-class', num_classes=num_classes)
        self.accuracy = torchmetrics.Accuracy(task = 'MULTICLASS', num_classes = num_classes)
        self.my_accuracy = MyAccuracy()
        self.F1Score = torchmetrics.F1Score(task = 'MULTICLASS', num_classes = num_classes)
    def forward(self,x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    def training_step(self, batch, batch_idx):
        loss, scores, y = self._common_step(batch, batch_idx)
        accuracy = self.accuracy(scores, y)
        f1score = self.F1Score(scores,y)
        self.log_dict(
            {
                'train_loss': loss,
                'accuracy': accuracy,
                'f1score': f1score
            }
            on_step= False,
            on_epoch = True,
            prog_bar = True
        )
        # if batch_idx % 100 == 0:
        #     x = x[:8]
        #     grid = torchvision.utils.make_grid(x.view(-1, 1,28,28))
        #     self.logger.experiment.add_image('mnist_images', grid,self.global_step)
        return self.log('loss', loss)
    def validation_step(self,batch, batch_idx):
        loss, scores, y = self._common_step(batch, batch_idx)
        return self.log('loss', loss)
    def test_step(self,batch, batch_idx):
        loss, scores, y = self._common_step(batch, batch_idx)
        return self.log('loss', loss)
    def _common_step(self, batch, batch_idx):
        x,y = batch
        x = x.reshape(x.shape[0], -1)
        predictions = self.forward(x)
        loss = self.loss(predictions, y)
        _, scores = predictions.max(1)
        return loss, scores, y
    def prediction_step(self, batch, batch_idx):
        x,y = batch
        x = x.reshape(x.shape[0], -1)
        predictions = self.forward(x)
        loss = self.loss(predictions, y)
        scores = torch.argmax(predictions, dim = 1)
        return scores
    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr = 0.001)

In [176]:
trainer = pl.Trainer(
    accelerator = 'cpu',
    devices = 1,
    precision = '16-mixed'
)
model = NN()
trainer.fit(model, train_dataloader, val_dataloader)
trainer.validate(model, val_dataloader)
trainer.test(model, test_dataloader)

C:\Users\ABDALAH\anaconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\accelerator_connector.py:479: You passed `Trainer(accelerator='cpu', precision='16-mixed')` but AMP with fp16 is not supported on CPU. Using `precision='bf16-mixed'` instead.
Using bfloat16 Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
C:\Users\ABDALAH\anaconda3\Lib\site-packages\pytorch_lightning\loops\utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.

  | Name        | Type               | Params | Mode 
-----------------------------------------------------------
0 | fc1         | Linear             | 39.2 K | train
1 | fc2         | Linear             | 510    | train


Sanity Checking: |                                                                               | 0/? [00:00<…

C:\Users\ABDALAH\anaconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:428: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Training: |                                                                                      | 0/? [00:00<…

C:\Users\ABDALAH\anaconda3\Lib\site-packages\pytorch_lightning\loops\optimization\automatic.py:134: `training_step` returned `None`. If this was on purpose, ignore this warning...


...

Validation: |                                                                                    | 0/? [00:00<…

Validation: |                                                                                    | 0/? [00:00<…


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

C:\Users\ABDALAH\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
from torchmetrics import Metric
class MyAccuracy(Metric):
    def __init__(self):
        super().__init__()
        self.add_state('total', default = torch.tensor(0))
        self.add_state('correct', default = torch.tensor(0))
    def update(self, target, preds):
        pred = torch.argmax(preds, dim = 1)
        assert pred == target, 'must be the same'
        self.correct += (pred == target).sum() 
        self.total += torch.numel() 
    def compute(self):
        return self.correct.float()/self.total.float()

In [ ]:
class MnistdataModule(pl.LightningModule):
    def __init__(self, batch_size=batch_size, num_workers=num_workers, data_dir = 'data'):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.data_dir = data_dir
    def prepare(self):
        datasets.MNIST(self.data_dir, download = True, train = True)
        datasets.MNIST(self.data_dir, download = True, train = False)
    def setup(self):
        entire_dataset = datasets.MNIST(
            root = self.data_dir,
            download = False,
            train = True
        )
        train_dataset, val_dataset = random_split(entire_dataset, [50000,10000])
        test_dataset = datasets.MNIST(
            root = self.data_dir,
            download = False,
            train = False
        )
        return train_dataset, val_dataset,test_dataset
    def train_dataloader(self):
        train_dataloader = DataLoader(
            self.data_dir,
            num_workers = self.num_workers,
            batch_size = self.batch_size,
            shuffle = True
        )
        return train_dataloader
    def val_dataloader(self):
        val_dataloader = torch.utils.data.DataLoader(
            self.data_dir,
            num_workers = self.num_workers,
            batch_size = self.batch_size,
            shuffle = True
        )
        return val_dataloader
    def test_dataloader(self):
        train_dataloader = torch.utils.data.DataLoader(
            self.data_dir,
            num_workers = self.num_workers,
            batch_size = self.batch_size,
            shuffle = True
        )
        return test_dataloader

In [ ]:
model = NN()
dm = MnistdataModule()
trainer = pl.Trainer(
    accelerator = 'cpu',
    precision = '16-mixed',
    devices = 1
)
trainer.fit(model, dm)
trainer.validate(model, dm)
trainer.test(model, dm)

In [ ]:
##train.py
from model import NN
from dataset import MnistdataModule
import config
from pytorch_lightning.loggers import TensorBoardLoggers
from pytorch_lightning.profiles import PytorchProfilers
from callbacks import EarlyStopping, MyPrintingCallback
if __name__=='__main__':
    logger = TensorBoardLoggers('tb_logs', name = 'mnist_model_v0')
    profiler = PytorchProfilers(on_trace_ready = torch.profiler.tensorboard_trace_handler("tb_logs/profiler0"), schedule = torch.profile.schedule(skip_first = 10, warmup =1 ,wait = 1, active = 20))
    dm = MnistdataModule()
    model = NN()
    trainer = pl.Trainer(callback = [MyPrintingCallback(), EarlyStopping(monitor = 'val_loss')])
    trainer.fit(model, dm)
    trainer.validate(model, dm)
    trainer.test(model, dm)

In [ ]:
##mycallback
from pytorch_lightning.callbacks import EarlyStopping, Callback
class MyPrintingCallback(Callback):
    def __init__(self):
        super().__init__()
    def on_train_start(self, trainer, pl_module):
        print('Starting to Train')
    def on_train_end(self, trainer, pl_module):
        print('Ending training')